# Day 8 — NumPy Fundamentals & Vectorization

This notebook covers NumPy Fundamentals, Memory Architecture, Vectorization, Broadcasting, and Array Slicing

## Validation goals
- Understand NumPy memory layout (contiguous blocks) versus standard Python lists (scattered pointers)
- Control data types explicitly via `dtype` and understand upcastings
- Inspect and manipulate array dimensions using `.shape` and `.reshape()`
- Contrast Python `for` loops with vectorized C-level operations
- Apply NumPy broadcasting rules (right-to-left alignment, incompatible shape failures)
- Execute vectorized financial calculations (`np.diff()` and percentage returns)
- Perform multi-dimensional array slicing


NumPy forms the backbone of numerical computing in Python. To understand why it is so much faster than standard Python structures, we must look at how data is stored in memory.

NumPy arrays are stored contiguously in memory, computer's CPU can fetch and process blocks of data in bulk using vectorized CPU instructions, bypassing the slow type-checking loop required by standard Python lists.

NumPy would typically convert both values to a common data type, usually a string type in this case, so that every element in the array has the same data type.

That process is called upcasting. Because NumPy arrays enforce homogeneity, mixed types are converted to the most flexible common denominator—often strings when text is involved.

### Guiding Question: 
If you wanted to create a NumPy array from a Python list of integers [1, 2, 3], but you wanted to force it to store them as floating-point numbers instead, how would you write the code using np.array()?

In [ ]:
import numpy as np

arr = np.array([1, 2, 3], dtype=float)

print(arr)
# [1. 2. 3.]

print(arr.dtype)
# float64

Explicitly setting dtype=float forces NumPy to allocate 64 bits per element, ensuring precision for numerical calculations.

### Guiding Question:

When working with financial datasets, 1-dimensional lists expand into 2-dimensional structures to represent rows and columns—such as historical prices for multiple assets over several days.

To inspect the structure of an array, NumPy provides attributes like .shape and .ndim. If you initialize a 2D array representing a table of numbers, what attribute would you query to find the exact count of rows and columns?

In [ ]:
import numpy as np

arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print(arr.shape)
# (2, 3)

### Guiding Question: 

If you wanted to reorganize arr from a (2, 3) shape into a 3-row by 2-column matrix, what method would you call on the array?

In [ ]:
arr = arr.reshape(3, 2)

print(arr)
# [[1 2]
#  [3 4]
#  [5 6]]

print(arr.shape)
# (3, 2)

When performing calculations on financial datasets—such as applying adjustments or computing metrics across thousands of rows—NumPy uses vectorization. Instead of iterating through elements one by one using a slow Python for loop, vectorization delegates the operation to optimized C code, executing calculations across blocks of memory simultaneously.

For instance, if you have an array arr = np.array([1, 2, 3]), running arr + 10 instantly adds 10 to every element without a loop.

Let’s apply this to a financial scenario. Imagine you have an array of stock prices prices = np.array([100.0, 105.0, 102.0]) and you want to subtract a flat trading fee of 2.5 from every price in the array at once.

How would you write the single-line vectorized expression to calculate the new prices?

In [ ]:
new_prices = prices - 2.5

## Guiding Question: 

If you multiply two 1D NumPy arrays of identical length (prices * volumes), how does NumPy align and process the elements in memory compared to doing the same operation with two standard Python lists?

In [ ]:
import numpy as np

prices = np.array([10, 20, 30])
volumes = np.array([2, 3, 4])

result = prices * volumes

print(result)
# [ 20  60 120]

# element wise multiplication

### Guiding Question: 

Why does a standard Python for loop incur a massive performance overhead compared to a vectorized NumPy operation when processing millions of numbers?

### Answer:

A standard Python for loop is slower than a vectorized NumPy operation primarily because Python executes the loop and its operations through its interpreter, while NumPy delegates numerical operations to optimized, compiled code written in languages such as C.

### Guiding Question:

Imagine you have a 2D array of stock prices with shape (3, 2) and a 1D array representing a daily adjustment factor with shape (2,). Without changing the shape of the smaller array manually, how does NumPy interpret and align the dimensions to apply that adjustment across every row?

### Answer:

NumPy uses broadcasting to automatically align the dimensions of the 1D adjustment array with the 2D stock prices array.
It aligns the dimensions from right to left. Since the adjustment array has shape (2,) and the stock prices array has shape (3, 2), NumPy treats the adjustment array as if it had shape (1, 2) and broadcasts it across all three rows.

### Guiding QUestion:

What behavior or error would you expect if you tried to broadcast a 1D array of shape (3,) across a 2D array of shape (3, 2)?

### Answer:

NumPy would raise a ValueError because the shapes (3, 2) and (3,) are incompatible for broadcasting. NumPy compares dimensions from right to left: the trailing dimensions are 2 and 3, which do not match and neither is 1, resulting in a ValueError.

### Guiding Question: 

If you pass prices into np.diff(), what will the resulting array look like in terms of values and length?

In [ ]:
import numpy as np

prices = np.array([100.0, 102.5, 101.0, 105.0])

differences = np.diff(prices)

print(differences)
# [ 2.5 -1.5  4. ]

### Guiding Question: 

If the array of differences has shape (3,) and we want to divide each element by the base price at the start of that interval (which corresponds to prices[:-1]), how would you write the vectorized expression to calculate daily percentage returns?

In [ ]:
daily_returns = np.diff(prices) / prices[:-1]

# or

daily_returns = (np.diff(prices) / prices[:-1]) * 100

Imagine we have a 2D NumPy array representing daily closing prices for 3 different assets over 4 trading days, giving it a shape of (3, 4):

Our goal is to extract a sub-matrix containing all rows, but only columns from index 1 to the end (dropping the first column).

How would you write the NumPy slicing expression using the colon : syntax for both dimensions ([row_selection, col_selection]) to extract this?

In [ ]:
prices_matrix = np.array([
    [100.0, 102.0, 101.5, 104.0],  # Asset 1
    [50.0,  51.0,  49.5,  52.0],   # Asset 2
    [200.0, 198.0, 202.0, 205.0]   # Asset 3
])

# Answer:

prices_matrix[:, 1:]